# 음악생성 재학습: 처음부터 충분히 학습하기

이전 모델은 학습 스텝이 300번뿐이라, 음표를 고를 때 후보가 평균 67개로 퍼져 거의 무작위로 연주했습니다.
이 노트북은 같은 Music Transformer 구조로 **처음부터 훨씬 오래** 학습합니다.

| 단계 | 데이터 | 목적 |
|---|---|---|
| 1단계 | MAESTRO 전체 (모든 작곡가) | 피아노 음악의 기본(화성, 리듬) 익히기 |
| 2단계 | 쇼팽 곡 | 쇼팽 스타일 익히기 |
| 3단계 | 캐롤 | 캐롤 분위기 익히기 |

**실행 방법**
1. 런타임 유형을 **GPU**로 바꿉니다.
2. 위에서부터 차례로 실행합니다. 1단계가 가장 오래 걸립니다.
3. 세션이 끊기면 **처음부터 다시 실행**하면 됩니다.
   토큰화 결과와 모델이 Drive `MyDrive/music_transformer_retrain`에 저장되어 있어, 끊긴 epoch부터 이어서 학습합니다.
4. 3단계 전에 캐롤 MIDI zip을 Colab `/content`에 올립니다.
5. 마지막 셀이 단계별 이어 연주와 학습 기록을 zip으로 내려받습니다.

**확인할 숫자**: 이전 모델의 검증 손실은 약 4.9였습니다. 손실이 낮을수록 다음 음을 더 정확히 맞힙니다.

## 1. 설치와 설정

In [ ]:
!pip install -qq midi_neural_processor
!pip install -qq keras_hub
!pip install -qq "keras>=3.6.0"
!pip install -qq pretty_midi

from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import random
import tempfile

import keras
import midi_neural_processor.processor as midi_tokenizer
import numpy as np
from keras import callbacks, layers, ops, optimizers, utils
from keras_hub import layers as hub_layers
from os import path

In [ ]:
event_range = midi_tokenizer.RANGE_NOTE_ON
event_range += midi_tokenizer.RANGE_NOTE_OFF
event_range += midi_tokenizer.RANGE_TIME_SHIFT
event_range += midi_tokenizer.RANGE_VEL
CONFIG = utils.Config(
    max_sequence_len=1024,   # 2048 -> 1024: 한 스텝 계산량이 약 4배 줄어 같은 시간에 훨씬 많이 학습
    embedding_dim=256,
    num_transformer_blocks=6,
    batch_size=8,
    token_pad=event_range,
    token_start_of_sentence=event_range + 1,
    token_end_of_sentence=event_range + 2,
    vocabulary_size=event_range + 3,
    seed=42,
)
utils.set_random_seed(CONFIG.seed)

# 모든 결과가 저장되는 Drive 폴더 (세션이 끊겨도 남음)
WORK = "/content/drive/MyDrive/music_transformer_retrain"
os.makedirs(WORK, exist_ok=True)
print("저장 폴더:", WORK)

## 2. 모델 구조 (원래 노트북과 같음)

In [ ]:
@keras.utils.register_keras_serializable()
class RelativeGlobalAttention(layers.Layer):
    """
    From Music Transformer (Huang et al., 2018)
    https://arxiv.org/abs/1809.04281
    """

    def __init__(self, num_heads, embedding_dim, max_sequence_len, **kwargs):
        super().__init__(**kwargs)
        self.key_length = None
        self.max_sequence_len = max_sequence_len
        self.relative_embedding = None
        self.num_heads = num_heads
        self.embedding_dim = embedding_dim
        self.head_dim = embedding_dim // num_heads
        self.query_dense = layers.Dense(int(self.embedding_dim))
        self.key_dense = layers.Dense(int(self.embedding_dim))
        self.value_dense = layers.Dense(int(self.embedding_dim))
        self.output_dense = layers.Dense(embedding_dim, name="output")

    def build(self, input_shape):
        self.query_length = input_shape[0][1]
        self.key_length = input_shape[1][1]
        self.relative_embedding = self.add_weight(
            (self.max_sequence_len, int(self.head_dim)), name="relative_embedding"
        )

    def _apply_dense_layer_and_split_heads(self, inputs, dense_layer):
        # Apply linear transformation
        inputs = dense_layer(inputs)
        new_shape = ops.shape(inputs)
        # Reshape to split by attention heads
        reshaped = ops.reshape(inputs, (new_shape[0], new_shape[1], self.num_heads, -1))
        # Transpose for head-first format
        return ops.transpose(reshaped, (0, 2, 1, 3))

    def call(self, inputs, mask=None):
        # Compute Q, K, V: Batch, head, sequence, features
        query = self._apply_dense_layer_and_split_heads(inputs[0], self.query_dense)
        key = self._apply_dense_layer_and_split_heads(inputs[1], self.key_dense)
        value = self._apply_dense_layer_and_split_heads(inputs[2], self.value_dense)

        # Compute scaled dot-product attention scores
        attention_scores = ops.matmul(query, ops.transpose(key, [0, 1, 3, 2]))

        # Compute relative positional encoding and combine with attention scores
        start_idx = max(0, self.max_sequence_len - ops.shape(query)[2])
        relative_embedding = self.relative_embedding[start_idx:, :]
        attention_scores += self._compute_attention_scores(query, relative_embedding)
        logits = attention_scores / ops.sqrt(self.head_dim)

        # Apply mask if provided
        if mask is not None:
            logits += ops.cast(mask, "float32") * -1e9

        # Compute attention weights
        attention_weights = ops.nn.softmax(logits, axis=-1)
        attention_output = ops.matmul(attention_weights, value)

        # Merge heads and apply final linear transformation
        merged_attention = ops.transpose(attention_output, (0, 2, 1, 3))
        merged_attention = ops.reshape(
            merged_attention, (ops.shape(merged_attention)[0], -1, self.embedding_dim)
        )
        output = self.output_dense(merged_attention)

        return output, attention_weights

    def _compute_attention_scores(self, query, relative_embedding):
        """
        Compute relative attention scores using positional encodings.
        """
        relative_scores = ops.einsum("bhld, md->bhlm", query, relative_embedding)
        relative_scores = self._apply_mask_to_relative_scores(relative_scores)
        return self._skew_attention_scores(relative_scores)

    def _apply_mask_to_relative_scores(self, scores):
        """
        Apply masking to relative positional scores to ignore future positions.
        """
        mask = ops.flip(
            ops.tri(scores.shape[-2], scores.shape[-1], dtype="float32"), axis=1
        )
        return mask * scores

    def _skew_attention_scores(self, scores):
        """
        Perform skewing operation to align relative attention scores with the sequence.
        """
        padded_scores = ops.pad(scores, ((0, 0), (0, 0), (0, 0), (1, 0)))
        padded_shape = ops.shape(padded_scores)
        reshaped_scores = ops.reshape(
            padded_scores, (-1, padded_shape[1], padded_shape[-1], padded_shape[-2])
        )
        skewed_scores = reshaped_scores[:, :, 1:, :]

        if self.key_length > self.query_length:
            size_diff = self.key_length - self.query_length
            return ops.pad(skewed_scores, [[0, 0], [0, 0], [0, 0], [0, size_diff]])
        else:
            return skewed_scores[:, :, :, : self.key_length]

In [ ]:
@keras.utils.register_keras_serializable()
class DecoderLayer(layers.Layer):
    def __init__(self, embedding_dim, num_heads, max_sequence_len, dropout=0.1):
        super(DecoderLayer, self).__init__()

        # Initialize attributes
        self.embedding_dim = embedding_dim
        self.num_heads = num_heads
        self.max_sequence_len = max_sequence_len

        # Initialize layers
        self.relative_global_attention_1 = RelativeGlobalAttention(
            num_heads, embedding_dim, max_sequence_len
        )

        self.feed_forward_network_pre = layers.Dense(self.embedding_dim // 2, "relu")
        self.feed_forward_network_pos = layers.Dense(self.embedding_dim)

        self.layer_normalization_1 = layers.LayerNormalization(epsilon=1e-6)
        self.layer_normalization_2 = layers.LayerNormalization(epsilon=1e-6)

        self.dropout_1 = layers.Dropout(dropout)
        self.dropout_2 = layers.Dropout(dropout)

    def call(self, inputs, mask=None, training=False):
        # Attention block. Inputs are (query, key, value)
        attention_out, attention_weights = self.relative_global_attention_1(
            (inputs, inputs, inputs), mask=mask
        )
        attention_out = self.dropout_1(attention_out, training=training)
        attention_out_normalized = self.layer_normalization_1(attention_out + inputs)

        ffn_out = self.feed_forward_network_pre(attention_out)
        ffn_out = self.feed_forward_network_pos(ffn_out)
        ffn_out = self.dropout_2(ffn_out, training=training)
        out = self.layer_normalization_2(attention_out_normalized + ffn_out)

        return out, attention_weights

In [ ]:
@keras.utils.register_keras_serializable()
class Decoder(layers.Layer):
    def __init__(
        self, embedding_dim, vocabulary_size, max_sequence_len, num_blocks, dropout
    ):
        super(Decoder, self).__init__()

        self.embedding_dim = embedding_dim
        self.num_blocks = num_blocks

        self.embedding = layers.Embedding(vocabulary_size, self.embedding_dim)
        self.positional_encoding = hub_layers.SinePositionEncoding()

        self.decode_layers = [
            DecoderLayer(
                embedding_dim, embedding_dim // 64, max_sequence_len, dropout=dropout
            )
            for _ in range(num_blocks)
        ]
        self.dropout = layers.Dropout(dropout)

    def call(self, inputs, mask=None, training=False, return_attention_weights=False):
        weights = []

        # Adding embedding and position encoding.
        x = self.embedding(inputs)
        x = x * ops.sqrt(ops.cast(self.embedding_dim, "float32"))
        x = x + self.positional_encoding(x)
        x = self.dropout(x, training=training)

        # Passing through the transformer blocks.
        for i in range(self.num_blocks):
            x, w = self.decode_layers[i](x, mask=mask, training=training)
            weights.append(w)
        if return_attention_weights:
            return x, weights
        return x

In [ ]:
@keras.utils.register_keras_serializable()
class MusicTransformerDecoder(keras.Model):
    def __init__(
        self,
        embedding_dim=CONFIG.embedding_dim,
        vocabulary_size=CONFIG.vocabulary_size,
        num_blocks=CONFIG.num_transformer_blocks,
        max_sequence_len=CONFIG.max_sequence_len,
        dropout=0.2,
    ):
        # Initialize attributes
        super(MusicTransformerDecoder, self).__init__()
        self.embedding_dim = embedding_dim
        self.vocabulary_size = vocabulary_size
        self.num_blocks = num_blocks
        self.max_sequence_len = max_sequence_len

        # Initialize layers
        # Transformer decoder
        self.decoder = Decoder(
            embedding_dim, vocabulary_size, max_sequence_len, num_blocks, dropout
        )
        # Output layer
        self.fc = layers.Dense(self.vocabulary_size, activation=None, name="output")

    @staticmethod
    def get_look_ahead_mask(max_sequence_len, inputs):
        sequence_length = min(max_sequence_len, inputs.shape[1])
        sequence_mask = ops.logical_not(
            ops.tri(sequence_length, sequence_length, dtype="bool")
        )

        inputs = ops.cast(inputs[:, None, None, :], "int32")
        output_pad_tensor = ops.ones_like(inputs) * CONFIG.token_pad
        decoder_output_mask = ops.equal(inputs, output_pad_tensor)
        return ops.cast(ops.logical_or(decoder_output_mask, sequence_mask), "int32")

    def call(self, inputs, training=False):
        mask = self.get_look_ahead_mask(self.max_sequence_len, inputs)
        decoding = self.decoder(
            inputs, mask=mask, training=training, return_attention_weights=False
        )
        return self.fc(decoding)

    # --- Sequence generation methods

    def generate(self, inputs: list, length=CONFIG.max_sequence_len, top_k=5):
        inputs = ops.convert_to_tensor([inputs])

        # Generate a new token using output distribution at given index
        def generate_token(inputs, end_idx):
            distribution = ops.stop_gradient(self.call(inputs)[0, end_idx])

            # Select the top-k tokens and their probabilities
            top_k_distribution, top_k_indices = ops.top_k(distribution, k=top_k)

            # Sample from the top-k probabilities
            new_token_idx = keras.random.categorical(top_k_distribution[None, :], 1)
            return ops.take(top_k_indices, new_token_idx[0])

        # Compute the number of tokens to add
        added_tokens = min(length, self.max_sequence_len - inputs.shape[1])
        progbar = utils.Progbar(added_tokens, unit_name="token", interval=5)

        # Pad the input sequence that will be filled with generated tokens
        out = ops.pad(inputs, ((0, 0), (0, added_tokens)), "constant", CONFIG.token_pad)

        # Generate tokens using top-k sampling
        for token_idx in range(inputs.shape[1] - 1, inputs.shape[1] - 1 + added_tokens):
            token = ops.cast(generate_token(out, end_idx=token_idx), out.dtype)
            out = ops.scatter_update(out, ((0, token_idx + 1),), token)
            progbar.add(1)

        return ops.convert_to_numpy(out[0])

    # --- Serialization methods

    def get_config(self):
        atts = ["embedding_dim", "vocabulary_size", "num_blocks", "max_sequence_len"]
        return {a: getattr(self, a) for a in atts}

    @classmethod
    def from_config(cls, config):
        return cls(**config)

In [ ]:
@keras.utils.register_keras_serializable()
def train_loss(y_true, y_pred):
    mask = ops.cast(ops.logical_not(ops.equal(y_true, CONFIG.token_pad)), "float32")
    y_true = ops.one_hot(ops.cast(y_true, "int32"), CONFIG.vocabulary_size)
    return ops.categorical_crossentropy(y_true, y_pred, from_logits=True) * mask

In [ ]:
@keras.utils.register_keras_serializable()
class CustomSchedule(optimizers.schedules.LearningRateSchedule):
    def __init__(self, embedding_dim, warmup_steps=4000):
        super(CustomSchedule, self).__init__()

        self.embedding_dim = embedding_dim
        self.warmup_steps = warmup_steps

        self._embedding_dim = ops.cast(self.embedding_dim, "float32")
        # Numerical stability adjustment on torch, which is less precise
        self._lr_adjust = 0.1 if keras.backend.backend() == "torch" else 1.0

    def get_config(self):
        return {"embedding_dim": self.embedding_dim, "warmup_steps": self.warmup_steps}

    def __call__(self, step):
        step_rsqrt = ops.rsqrt(ops.cast(step, "float32"))
        warmup_adjust = step * (self.warmup_steps**-1.5)
        output = ops.rsqrt(self._embedding_dim) * ops.minimum(step_rsqrt, warmup_adjust)
        return self._lr_adjust * output

## 3. 데이터

In [ ]:
class MidiDataset(utils.PyDataset):
    """Yields random (input, target) windows. `steps` fixes how many batches make one epoch."""

    def __init__(self, encoded_midis, batch_size=CONFIG.batch_size,
                 max_sequence_len=CONFIG.max_sequence_len, steps=None):
        super().__init__()
        self.batch_size = min(batch_size, len(encoded_midis))
        self.max_sequence_len = max_sequence_len
        self.encoded_midis = encoded_midis
        self.steps = steps or -(-len(encoded_midis) // self.batch_size)

    def __len__(self):
        return self.steps

    def __getitem__(self, idx):
        batch = random.sample(self.encoded_midis, k=self.batch_size)
        data = np.array([self._get_sequence(m, self.max_sequence_len + 1) for m in batch])
        return data[:, :-1], data[:, 1:]

    def _get_sequence(self, data, max_length):
        if len(data) > max_length:
            start = random.randrange(0, len(data) - max_length)
            data = data[start : start + max_length]
        elif len(data) < max_length:
            data = np.append(data, CONFIG.token_end_of_sentence)
        if len(data) < max_length:
            data = np.concatenate((data, np.full(max_length - len(data), CONFIG.token_pad)))
        return np.asanyarray(data, dtype="int32")

In [ ]:
# MAESTRO 내려받기 + 토큰화 (토큰화 결과는 Drive에 저장해 두고, 다음 세션부터는 바로 불러옴)
import glob, zipfile, urllib.request
import pandas as pd
from multiprocessing import Pool

MAESTRO_ZIP = "/content/maestro-v3.0.0-midi.zip"
if not os.path.exists("/content/maestro"):
    urllib.request.urlretrieve(
        "https://storage.googleapis.com/magentadata/datasets/maestro/v3.0.0/maestro-v3.0.0-midi.zip", MAESTRO_ZIP)
    with zipfile.ZipFile(MAESTRO_ZIP) as z:
        z.extractall("/content/maestro")
meta = pd.read_csv(glob.glob("/content/maestro/**/maestro-v3.0.0.csv", recursive=True)[0])
base = os.path.dirname(glob.glob("/content/maestro/**/maestro-v3.0.0.csv", recursive=True)[0])

def encode_task(p):
    import midi_neural_processor.processor as tok
    try:
        return np.array(tok.encode_midi(p), dtype=np.int32)
    except Exception:
        return None

def tokens_for(name, file_paths):
    cache = f"{WORK}/tokens_{name}.npz"
    if os.path.exists(cache):
        z = np.load(cache)
        return [z[k] for k in z.files]
    with Pool(max(1, os.cpu_count() - 1)) as pool:
        out = [t for t in pool.imap_unordered(encode_task, file_paths) if t is not None and len(t) > 64]
    np.savez(cache, *out)
    return out

def split_paths(rows):
    return [os.path.join(base, f) for f in rows["midi_filename"]]

is_chopin = meta["canonical_composer"].str.contains("Chopin", na=False)
maestro_train = tokens_for("maestro_train", split_paths(meta[meta.split == "train"]))
maestro_val   = tokens_for("maestro_val",   split_paths(meta[meta.split == "validation"]))
chopin_train  = tokens_for("chopin_train",  split_paths(meta[is_chopin & (meta.split != "validation")]))
chopin_val    = tokens_for("chopin_val",    split_paths(meta[is_chopin & (meta.split == "validation")]))
print(f"MAESTRO 학습 {len(maestro_train)}곡 / 검증 {len(maestro_val)}곡, 쇼팽 학습 {len(chopin_train)}곡 / 검증 {len(chopin_val)}곡")

## 4. 학습 (단계마다 Drive에 저장, 끊기면 이어서)

In [ ]:
# 이어서 학습할 수 있는 학습 함수
# - 매 epoch이 끝날 때마다 모델(옵티마이저 상태 포함)과 진행한 epoch 수를 Drive에 저장
# - 세션이 끊겨 다시 실행하면 저장된 지점부터 이어서 학습
# - 검증 손실이 가장 낮은 모델은 따로 *_best.keras로 저장
import json

class SaveProgress(callbacks.Callback):
    def __init__(self, stage):
        super().__init__()
        self.stage = stage
    def on_epoch_end(self, epoch, logs=None):
        self.model.save(f"{WORK}/{self.stage}_latest.keras")
        path_ = f"{WORK}/{self.stage}_progress.json"
        prev = json.load(open(path_)) if os.path.exists(path_) else {}
        history = prev.get("history", []) + [{"epoch": epoch + 1, **{k: float(v) for k, v in (logs or {}).items()}}]
        best = min(h["val_loss"] for h in history if "val_loss" in h)
        with open(path_, "w") as f:
            json.dump({"epoch": epoch + 1, "best_val_loss": best, "history": history}, f)

def run_stage(stage, make_model, train_tokens, val_tokens, epochs, steps_per_epoch, lr):
    latest, prog = f"{WORK}/{stage}_latest.keras", f"{WORK}/{stage}_progress.json"
    saved = json.load(open(prog)) if os.path.exists(prog) else {}
    done, best = saved.get("epoch", 0), saved.get("best_val_loss")
    if done >= epochs:
        print(f"[{stage}] 이미 {epochs} epoch 완료")
        return keras.models.load_model(f"{WORK}/{stage}_best.keras")
    if done > 0:
        print(f"[{stage}] {done} epoch부터 이어서 학습")
        model = keras.models.load_model(latest)
    else:
        model = make_model()
        model.compile(optimizer=optimizers.Adam(lr, beta_1=0.9, beta_2=0.98, epsilon=1e-9), loss=train_loss)
    model.fit(
        MidiDataset(train_tokens, steps=steps_per_epoch),
        validation_data=MidiDataset(val_tokens, steps=20),
        epochs=epochs, initial_epoch=done, verbose=2,
        callbacks=[SaveProgress(stage),
                   callbacks.ModelCheckpoint(f"{WORK}/{stage}_best.keras", save_best_only=True,
                                             initial_value_threshold=best)],
    )
    return keras.models.load_model(f"{WORK}/{stage}_best.keras")

### 1단계: MAESTRO 전체

In [ ]:
# 1단계: MAESTRO 전체(모든 작곡가)로 처음부터 학습
# 40 epoch x 500 스텝 = 20,000 스텝 (예전 학습은 300 스텝)
# 학습률은 노트북의 CustomSchedule(4,000 스텝 워밍업)을 그대로 사용
stage1_model = run_stage(
    "stage1_maestro",
    make_model=lambda: MusicTransformerDecoder(max_sequence_len=CONFIG.max_sequence_len),
    train_tokens=maestro_train, val_tokens=maestro_val,
    epochs=40, steps_per_epoch=500, lr=CustomSchedule(CONFIG.embedding_dim),
)

### 2단계: 쇼팽

In [ ]:
# 2단계: 쇼팽 곡만으로 추가 학습 (쇼팽 스타일 익히기)
# 1단계 모델에서 시작하므로 학습률을 낮게(1e-4) 둠
stage2_model = run_stage(
    "stage2_chopin",
    make_model=lambda: keras.models.load_model(f"{WORK}/stage1_maestro_best.keras"),
    train_tokens=chopin_train, val_tokens=chopin_val,
    epochs=15, steps_per_epoch=300, lr=1e-4,
)

### 3단계: 캐롤

In [ ]:
# 3단계 준비: 캐롤 MIDI zip을 Colab /content에 올린 뒤 실행
# 평가용 캐롤 1곡은 학습에서 빼서, 외운 결과가 아닌지 확인할 수 있게 함
CAROL_ZIP = "/content/Midi_캐롤_인공지능자료분석-20251124T124057Z-1-001.zip"
TEST_CAROL = "2. Christmas Without you.mid"
with zipfile.ZipFile(CAROL_ZIP) as z:
    z.extractall("/content/carols")
carol_paths = sorted(glob.glob("/content/carols/**/*.mid", recursive=True))
test_path = [p for p in carol_paths if p.endswith(TEST_CAROL)][0]
carols = [t for t in (encode_task(p) for p in carol_paths if p != test_path) if t is not None and len(t) > 64]
random.seed(0)
random.shuffle(carols)
n_val = max(2, len(carols) // 8)
carol_val, carol_train = carols[:n_val], carols[n_val:]
print(f"캐롤 학습 {len(carol_train)}곡, 검증 {len(carol_val)}곡, 평가용 1곡 제외")

In [ ]:
# 3단계: 쇼팽 모델에 캐롤로 추가 학습
# 곡 수가 적어 외워 버리기 쉬우므로 학습률을 더 낮게(5e-5), epoch도 짧게
stage3_model = run_stage(
    "stage3_carol",
    make_model=lambda: keras.models.load_model(f"{WORK}/stage2_chopin_best.keras"),
    train_tokens=carol_train, val_tokens=carol_val,
    epochs=20, steps_per_epoch=50, lr=5e-5,
)

## 5. 단계별 이어 연주 비교

In [ ]:
# 단계별 모델로 같은 캐롤을 이어 연주해 비교
# - 캐롤 앞부분 384토큰을 들려주고 640토큰 생성 (문맥 길이 1024 안에서)
# - TOP_K 후보 중에서 샘플링. 학습이 충분하면 40 정도가 자연스러움
#   음표가 안 나오면 TOP_K를 키워 다시 실행
import shutil
import pretty_midi
from pathlib import Path
from google.colab import files

TOP_K = 40
SEED_TOKENS, NEW_TOKENS = 384, 640
out_dir = Path("/content/retrain_compare")
out_dir.mkdir(exist_ok=True)
seed = list(midi_tokenizer.encode_midi(test_path)[:SEED_TOKENS])
midi_tokenizer.decode_midi(np.array(seed), str(out_dir / "prompt.mid"))

def continue_carol(model_path, name):
    m = keras.models.load_model(model_path)
    keras.utils.set_random_seed(42)
    out = [int(t) for t in m.generate(seed, length=NEW_TOKENS, top_k=TOP_K) if t < CONFIG.token_pad]
    p = out_dir / f"{name}_generated.mid"
    midi_tokenizer.decode_midi(np.array(out[len(seed):]), str(p))
    n = sum(len(i.notes) for i in pretty_midi.PrettyMIDI(str(p)).instruments)
    print(f"{name}: 새 음표 {n}개")

continue_carol(f"{WORK}/stage1_maestro_best.keras", "stage1_maestro")
continue_carol(f"{WORK}/stage2_chopin_best.keras", "stage2_chopin")
continue_carol(f"{WORK}/stage3_carol_best.keras", "stage3_carol")

for stage in ("stage1_maestro", "stage2_chopin", "stage3_carol"):
    shutil.copy(f"{WORK}/{stage}_progress.json", out_dir)   # 학습 기록도 함께
shutil.make_archive("/content/retrain_compare", "zip", out_dir)
files.download("/content/retrain_compare.zip")

## 6. 쇼팽 샘플 5곡 생성

학습이 끝난 뒤에는 1~3번 섹션만 실행하고 바로 이 셀로 내려와도 됩니다.
(4·5번은 건너뛰어도 Drive에 저장된 2단계 모델을 불러옵니다.)

In [ ]:
# 쇼팽 샘플 5곡: 2단계(쇼팽) 모델로 생성
# - 학습에 쓰지 않은 쇼팽 검증곡 5곡의 앞부분 384토큰을 들려주고 640토큰 생성
# - 같은 곡의 실제 다음 부분도 함께 저장해, 베꼈는지 비교할 수 있게 함
import shutil
import pretty_midi
from pathlib import Path
from google.colab import files

TOP_K = 40
SEED_TOKENS, NEW_TOKENS = 384, 640
out_dir = Path("/content/chopin_samples")
out_dir.mkdir(exist_ok=True)
model = keras.models.load_model(f"{WORK}/stage2_chopin_best.keras")

long_pieces = [i for i, t in enumerate(chopin_val) if len(t) > SEED_TOKENS + NEW_TOKENS]
picked = random.Random(0).sample(long_pieces, 5)
for k, i in enumerate(picked, 1):
    piece = [int(t) for t in chopin_val[i]]
    seed = piece[:SEED_TOKENS]
    keras.utils.set_random_seed(k)
    out = [int(t) for t in model.generate(seed, length=NEW_TOKENS, top_k=TOP_K) if t < CONFIG.token_pad]
    midi_tokenizer.decode_midi(np.array(seed), str(out_dir / f"sample{k}_prompt.mid"))
    midi_tokenizer.decode_midi(np.array(out[SEED_TOKENS:]), str(out_dir / f"sample{k}_generated.mid"))
    midi_tokenizer.decode_midi(np.array(piece[SEED_TOKENS:SEED_TOKENS + NEW_TOKENS]), str(out_dir / f"sample{k}_original_next.mid"))
    n = sum(len(x.notes) for x in pretty_midi.PrettyMIDI(str(out_dir / f"sample{k}_generated.mid")).instruments)
    print(f"샘플 {k} (검증곡 {i}번): 새 음표 {n}개")

shutil.make_archive("/content/chopin_samples", "zip", out_dir)
files.download("/content/chopin_samples.zip")